In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('SalarySurvey2021.csv')
print('The original dataset contains', df.shape[0], 'rows and', df.shape[1], 'columns.')

The original dataset contains 27985 rows and 18 columns.


## Before analysing any dataset, it needs to be cleaned.
Data Cleaning consists of several essential tasks and techniques, including -
* Handling missing values
* Outlier identification
* Data type conversion
* Standardization and normalization
* Deduplication
* Addressing inconsistent data, formatting, typos and spelling errors

The column names will be renamed to make them more simplified. This is done to make the column names more clean, intuitive and easy to work with during analysis.

In [ ]:
for i, col in enumerate(df.columns):
    print(i, col)

0 Timestamp
1 How old are you?
2 What industry do you work in?
3 Job title
4 If your job title needs additional context, please clarify here:
5 What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)
6 How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.
7 Please indicate the currency
8 If "Other," please indicate the currency here: 
9 If your income needs additional context, please provide it here:
10 What country do you work in?
11 If you're in the U.S., what state do you work in?
12 What city do you work in?
13 How many years of professional work experience do you have overall?
14 How many years of professional work experience do you have in your field?
15 What is your highest 

In [ ]:
df.columns.values[1] = 'Age'
df.columns.values[2] = 'Industry'
df.columns.values[3] = 'Job_Title'
df.columns.values[4] = 'Job_Title_Context'
df.columns.values[5] = 'Annual_Salary'
df.columns.values[6] = 'Extra_Compensation'
df.columns.values[7] = 'Currency'
df.columns.values[8] = 'Currency_Other'
df.columns.values[9] = 'Income_Context'
df.columns.values[10] = 'Country'
df.columns.values[11] = 'U.S. State'
df.columns.values[12] = 'City'
df.columns.values[13] = 'Years_Experience_Overall'
df.columns.values[14] = 'Years_Experience_Field'
df.columns.values[15] = 'Highest_Education'
df.columns.values[16] = 'Gender'
df.columns.values[17] = 'Race'

As the dataset is from a 2021 survey, the entries from the years other than 2021 will be removed.

In [ ]:
df = df[df['Timestamp'].str.contains('2021')]

The 'Currency' and 'Currency_Other' columns can be merged for simplicity.

In [ ]:
df['Currency'] = np.where(df['Currency'] == 'Other', df['Currency_Other'], df['Currency'])
df.drop('Currency_Other', axis=1, inplace=True)

The entries with null values in the 'Industry' column are removed.

In [ ]:
df.dropna(subset=['Industry'], inplace=True)

For ease of analysis, I will only keep the entries which contain values from the form checkboxes for the 'Industry' field.

In [ ]:
# List of categories to keep
categories = ['Accounting, Banking & Finance','Agriculture or Forestry','Art & Design',
              'Business or Consulting',
              'Computing or Tech',
              'Education (Primary/Secondary)','Education (Higher Education)','Engineering or Manufacturing','Entertainment',
              'Government and Public Administration',
              'Health care','Hospitality & Events',
              'Insurance',
              'Law','Law Enforcement & Security','Leisure, Sport & Tourism',
              'Marketing, Advertising & PR','Media & Digital',
              'Nonprofits',
              'Property or Construction',
              'Recruitment or HR','Retail',
              'Sales','Social Work',
              'Transport or Logistics',
              'Utilities & Telecommunications']

# Filter DataFrame to only keep rows with category match
df = df[df['Industry'].isin(categories)]

Normalising the 'Country' column through the SequenceMatcher imported from the difflib library. A function is created which is used to compare the similarity ratios between each value in the 'Country' column and the values in the 'correct_names' array. This 'correct_names' array contains the correct names of all the countries.

Note that this method isn't 100% accurate but it is efficient. With large datasets, it is not possible to go through each entry to normalise the values in a specific column.

In [ ]:
from difflib import SequenceMatcher

df['Country'] = df['Country'].str.lower() # convert to lowercase
df['Country'] = df['Country'].str.replace('.', '', regex=False) # remove periods

#let's specify correct country names
country_names = {"Afghanistan","Albania","Algeria","Andorra","Angola","Anguilla","Antigua & Barbuda","Argentina","Armenia","Aruba","Australia",
                 "Austria","Azerbaijan",
                 "Bahamas","Bahrain","Bangladesh","Barbados","Belarus","Belgium","Belize","Benin","Bermuda","Bhutan","Bolivia",
                 "Bosnia & Herzegovina","Botswana","Brazil","British Virgin Islands","Brunei","Bulgaria","Burkina Faso","Burundi",
                 "Cambodia","Cameroon","Cape Verde","Cayman Islands","Chad","Chile","China","Colombia","Congo","Cook Islands","Costa Rica",
                 "Cote D Ivoire","Croatia","Cuba","Cyprus","Czech Republic",
                 "Denmark","Djibouti","Dominica","Dominican Republic",
                 "England","Ecuador","Egypt","El Salvador","Equatorial Guinea","Estonia","Ethiopia",
                 "Falkland Islands","Faroe Islands","Fiji","Finland","France","French Polynesia","French West Indies",
                 "Gabon","Gambia","Georgia","Germany","Ghana","Gibraltar","Greece","Greenland","Grenada","Guam","Guatemala","Guernsey","Guinea",
                 "Guinea Bissau","Guyana",
                 "Haiti","Honduras","Hong Kong","Hungary",
                 "Iceland","India","Indonesia","Iran","Iraq","Ireland","Isle of Man","Israel","Italy",
                 "Jamaica","Japan","Jersey","Jordan",
                 "Kazakhstan","Kenya","Kuwait","Kyrgyz Republic",
                 "Laos","Latvia","Lebanon","Lesotho","Liberia","Libya","Liechtenstein","Lithuania","Luxembourg",
                 "Macau","Macedonia","Madagascar","Malawi","Malaysia","Maldives","Mali","Malta","Mauritania","Mauritius","Mexico","Moldova",
                 "Monaco","Mongolia","Montenegro","Montserrat","Morocco","Mozambique",
                 "Northen Ireland","Namibia","Nepal","Netherlands","Netherlands Antilles","New Caledonia","New Zealand","Nicaragua","Niger",
                 "Nigeria","Norway",
                 "Oman",
                 "Pakistan","Palestine","Panama","Papua New Guinea","Paraguay","Peru","Philippines","Poland","Portugal","Puerto Rico",
                 "Qatar",
                 "Reunion","Romania","Russia","Rwanda",
                 "Scotland","Saint Pierre & Miquelon","Samoa","San Marino","Satellite","Saudi Arabia","Senegal","Serbia","Seychelles",
                 "Sierra Leone", "Singapore","Slovakia","Slovenia","South Africa","South Korea","Spain","Sri Lanka","St Kitts & Nevis",
                 "St Lucia","St Vincent","St. Lucia","Sudan","Suriname","Swaziland","Sweden","Switzerland","Syria",
                 "Taiwan","Tajikistan","Tanzania","Thailand","Timor L'Este","Togo","Tonga","Trinidad & Tobago","Tunisia","Turkey",
                 "Turkmenistan","Turks & Caicos",
                 "Uganda","Ukraine","United Arab Emirates","United Kingdom","UK","United States","USA","Uruguay","Uzbekistan",
                 "Venezuela","Vietnam","US Virgin Islands",
                 "Wales",
                 "Yemen",
                 "Zambia","Zimbabwe"};

country_names = {name.lower() for name in country_names} # convert to lowercase

#let's specify the function that select the most similar word
def get_most_similar(word,wordlist):
  top_similarity = 0.25 # set to 0.25 to ensure some similarity
  most_similar_word = word
  for candidate in wordlist:
    similarity = SequenceMatcher(None,word,candidate).ratio()
    if similarity > top_similarity:
      top_similarity = similarity
      most_similar_word = candidate

  return most_similar_word

#now apply this function over 'country' column in dataframe
df['Country'] = df['Country'].apply(lambda x: get_most_similar(x, country_names))

After checking the unique values in the 'Country' column, there are some values that can be grouped up. There are also a few instances of the wrong information being entered. These entries need to be removed.

In [ ]:
# Capitalizing the first letter of each word in the 'Country' column
df['Country'] = df['Country'].str.title()

unique_values = df['Country'].unique().tolist()
unique_values

['United States',
 'United Kingdom',
 'Usa',
 'Uganda',
 'Uk',
 'Scotland',
 'Netherlands',
 'Spain',
 'Finland',
 'France',
 'England',
 'Germany',
 'Ireland',
 'India',
 'Australia',
 'Argentina',
 'Denmark',
 'Algeria',
 'Switzerland',
 'Bermuda',
 'Malaysia',
 'Mexico',
 'South Africa',
 'Belgium',
 'Northen Ireland',
 'Sweden',
 'Hong Kong',
 'Kuwait',
 'Norway',
 'Sri Lanka',
 'Montserrat',
 'Us Virgin Islands',
 "We Don'T Get Raises, We Get Quarterly Bonuses, But They Periodically Asses Income In The Area You Work, So I Got A Raise Because A 3Rd Party Assessment Showed I Was Paid Too Little For The Area We Were Located",
 'Greece',
 'Japan',
 'Bahrain',
 'Austria',
 'Brazil',
 'San Marino',
 'Colombia',
 'Iceland',
 'Hungary',
 'Luxembourg',
 'Guinea Bissau',
 'New Zealand',
 'Trinidad & Tobago',
 'Cayman Islands',
 'China',
 'Latvia',
 'Puerto Rico',
 'Isle Of Man',
 'Rwanda',
 'United Arab Emirates',
 'Romania',
 'Serbia',
 'Philippines',
 'Poland',
 'Czech Republic',
 'Italy'

In [ ]:
country_mapping = {
    'United States': 'USA',
    'Usa': 'USA',
    '🇺🇸 ': 'USA',
    'I Work For A Uae-Based Organization, Though I Am Personally In The Us': 'USA', # I am mapping the 'USA' value to the original value
    'United Kingdom': 'UK',
    'Uk': 'UK',
    'Scotland': 'UK',
    'Wales': 'UK',
    'Northern Ireland': 'UK'
}

# Use the `replace` method to replace values in the 'Country' column
df['Country'] = df['Country'].replace(country_mapping, regex=False)

# Removing entries that have irrelevant information in the 'Country' column
df = df[df['Country'].str.len() <= 30]

In [ ]:
unique_values = df['Country'].unique().tolist()
unique_values

['USA',
 'UK',
 'Uganda',
 'Netherlands',
 'Spain',
 'Finland',
 'France',
 'England',
 'Germany',
 'Ireland',
 'India',
 'Australia',
 'Argentina',
 'Denmark',
 'Algeria',
 'Switzerland',
 'Bermuda',
 'Malaysia',
 'Mexico',
 'South Africa',
 'Belgium',
 'Northen Ireland',
 'Sweden',
 'Hong Kong',
 'Kuwait',
 'Norway',
 'Sri Lanka',
 'Montserrat',
 'Us Virgin Islands',
 'Greece',
 'Japan',
 'Bahrain',
 'Austria',
 'Brazil',
 'San Marino',
 'Colombia',
 'Iceland',
 'Hungary',
 'Luxembourg',
 'Guinea Bissau',
 'New Zealand',
 'Trinidad & Tobago',
 'Cayman Islands',
 'China',
 'Latvia',
 'Puerto Rico',
 'Isle Of Man',
 'Rwanda',
 'United Arab Emirates',
 'Romania',
 'Serbia',
 'Philippines',
 'Poland',
 'Czech Republic',
 'Italy',
 'Afghanistan',
 'Israel',
 'Russia',
 'Chad',
 'Papua New Guinea',
 'Grenada',
 'Mauritania',
 'Tanzania',
 'Taiwan',
 'Cambodia',
 'Vietnam',
 'Yemen',
 'Singapore',
 'South Korea',
 'Thailand',
 'Bangladesh',
 'Lithuania',
 'Indonesia',
 'Cuba',
 'Slovenia',


The missing values in the 'Extra_Compensation' column should be replaced with '0'.

In [ ]:
df['Extra_Compensation'].fillna(0, inplace=True)

The data type of the 'Annual_Salary' and 'Extra_Compensation' columns should be changed to int.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 25392 entries, 0 to 27596
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Timestamp                 25392 non-null  object 
 1   Age                       25392 non-null  object 
 2   Industry                  25392 non-null  object 
 3   Job_Title                 25392 non-null  object 
 4   Job_Title_Context         6596 non-null   object 
 5   Annual_Salary             25392 non-null  object 
 6   Extra_Compensation        25392 non-null  float64
 7   Currency                  25389 non-null  object 
 8   Income_Context            2729 non-null   object 
 9   Country                   25392 non-null  object 
 10  U.S. State                20852 non-null  object 
 11  City                      25326 non-null  object 
 12  Years_Experience_Overall  25392 non-null  object 
 13  Years_Experience_Field    25392 non-null  object 
 14  Highes

In [ ]:
df['Annual_Salary'] = df['Annual_Salary'].str.replace(',','')
df['Annual_Salary'] = df['Annual_Salary'].astype('int64')

df['Extra_Compensation'] = df['Extra_Compensation'].astype('int64')

I've then exported the clean dataframe to a .csv file.

In [ ]:
df.to_csv('ManagerSalarySurvey2021_updated.csv', index=False)